# MegatronBridge API — Colab Quickstart

This notebook runs the full MegatronBridge API stack on a Colab GPU.

**Verified environment (April 2026)**
```
Python:  3.12.13
PyTorch: 2.10.0+cu128
CUDA:    12.8 toolkit  (driver reports 13.0)
GPU:     NVIDIA RTX PRO 6000 Blackwell Server Edition (95.6 GB)
```

## ⚠️ Known Limitation — transformer-engine on Colab (April 2026)

Colab's Blackwell runtime ships **PyTorch 2.10.0**, which is too new for any
pre-built `transformer-engine` wheel. All available wheels were compiled against
older PyTorch versions and fail at runtime with:

```
undefined symbol: _ZNK3c104cuda10CUDAStream5queryEv
```

This is a C++ ABI mismatch between the TE shared library and PyTorch 2.10.0.

### Available fixes

| Fix | Time | Notes |
|-----|------|-------|
| **Build from source** (used in this notebook) | ~20 min | Compiles TE against your exact PyTorch + CUDA — always works |
| **Downgrade PyTorch to 2.6 or 2.7** | ~5 min | Pre-built TE wheels exist for those versions, but downgrading PyTorch in Colab is risky and may break other packages |
| **Use NeMo NGC container** | N/A | The production path (`nvcr.io/nvidia/nemo:25.02`) has everything pre-compiled together — no ABI issues. Not available directly in Colab. |
| **Wait for TE to release PyTorch 2.10 wheels** | N/A | Not actionable yet |

The build-from-source approach is slow but reliable. The **API layer** (HTTP endpoints,
WebSocket streaming, auth, job queue) is fully verified without needing transformer-engine.
transformer-engine is only needed when the GPU worker actually runs SDK operations.

**Required runtime: H100 or A100**  
Runtime → Change runtime type → Hardware accelerator: H100

[![GitHub](https://img.shields.io/badge/GitHub-Nvidia--megatron--bridge--API-black)](https://github.com/Doondi-Ashlesh/Nvidia-megatron-bridge-API)

## Step 1 — Verify environment

In [ ]:
import sys
import subprocess
import torch

print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print()

py_ok   = sys.version_info >= (3, 12)
pt_ok   = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) >= (2, 7)
cuda_ok = torch.version.cuda is not None and float(torch.version.cuda) >= 12.8

print(f"Python ≥ 3.12:  {'✅' if py_ok   else '❌'}")
print(f"PyTorch ≥ 2.7:  {'✅' if pt_ok   else '❌'}")
print(f"CUDA ≥ 12.8:    {'✅' if cuda_ok else '❌'}")

if not all([py_ok, pt_ok, cuda_ok]):
    raise RuntimeError("Requirements not met. Switch to H100/A100 runtime.")

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print()
print(result.stdout)

## Step 2 — Clone repository

In [ ]:
import os

if not os.path.exists("/content/Nvidia-megatron-bridge-API"):
    !git clone https://github.com/Doondi-Ashlesh/Nvidia-megatron-bridge-API /content/Nvidia-megatron-bridge-API

os.chdir("/content/Nvidia-megatron-bridge-API")
print("Working directory:", os.getcwd())

## Step 3 — Install dependencies

Run cells 3a → 3b → 3c → 3d in order. Each has a comment explaining why it's needed.

In [ ]:
# 3a — API server dependencies (CPU-only, no conflicts)
!pip install -e ".[dev]" -q
print("✅ API server dependencies installed")

In [ ]:
# 3b — megatron-bridge
# Use --no-deps to avoid megatron-core[dev,mlm] pulling in dev dependencies
# that conflict with Colab's environment. Install megatron-core separately
# without the dev extras.
!pip install setuptools pybind11 wheel_stub -q
!pip install megatron-bridge --no-deps -q
!pip install megatron-core -q
print("✅ megatron-bridge installed")

In [ ]:
# 3c — transformer-engine (build from source)
#
# WHY BUILD FROM SOURCE:
# Colab ships PyTorch 2.10.0 which is too new for any pre-built TE wheel.
# All pre-built wheels (PyPI, NVIDIA PyPI, core_cu12, core_cu13) fail with:
#   'undefined symbol: _ZNK3c104cuda10CUDAStream5queryEv'
# This is a C++ ABI mismatch between the wheel's compiled PyTorch version
# and 2.10.0. Building from source compiles TE against the exact PyTorch
# headers that are installed, so there is no ABI mismatch.
#
# This takes ~20 minutes. Go make a coffee.
#
# ALTERNATIVE: downgrade PyTorch to 2.6 or 2.7 (pre-built wheels exist
# for those) — but downgrading PyTorch in Colab is risky and may break
# other packages.
print("Building transformer-engine from source (~20 min)...")
!pip install ninja -q
!NVTE_FRAMEWORK=pytorch pip install git+https://github.com/NVIDIA/TransformerEngine.git@stable -q
print("✅ transformer-engine built from source")

In [ ]:
# 3d — pin transformers
# megatron-bridge 0.3.1 requires transformers<5.0.0.
# Colab ships 5.0.0 exactly, which is incompatible.
!pip install "transformers<5.0.0" -q
print("✅ transformers pinned to <5.0.0")

## Step 4 — Configure environment

In [ ]:
import os

os.environ["CHECKPOINTS_ROOT"] = "/content/checkpoints"
os.environ["LOGS_ROOT"]        = "/content/logs"
os.environ["DATABASE_URL"]     = "sqlite+aiosqlite:////content/megatronbridge.db"
os.environ["LOG_LEVEL"]        = "INFO"

# CUDA library path — must be set before starting uvicorn so the worker
# subprocess inherits it. Without this, the worker fails with:
#   'libcublas.so.12: cannot open shared object file'
os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/cuda-12.8/targets/x86_64-linux/lib:"
    "/usr/local/lib/python3.12/dist-packages/nvidia/cublas/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

!mkdir -p /content/checkpoints /content/logs

print("✅ Environment configured")
print(f"  Checkpoints:  {os.environ['CHECKPOINTS_ROOT']}")
print(f"  Logs:         {os.environ['LOGS_ROOT']}")
print(f"  Database:     {os.environ['DATABASE_URL']}")

## Step 5 — Start the API server

In [ ]:
import subprocess
import time
import urllib.request

server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8000", "--log-level", "info"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),  # passes LD_LIBRARY_PATH to all child processes
)

print("Starting API server", end="")
for _ in range(15):
    time.sleep(1)
    print(".", end="", flush=True)
    try:
        resp = urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print(f"\n✅ Server is up: {resp.read().decode()}")
        break
    except Exception:
        continue
else:
    out = server_proc.stdout.read1(4096)
    print(f"\n❌ Server did not start:\n{out.decode()}")

In [ ]:
import requests
import json

BASE = "http://localhost:8000"

resp = requests.get(f"{BASE}/v1/system/info")
print(json.dumps(resp.json(), indent=2))
# peak_tflops is populated for H100, A100, RTX PRO 6000 Blackwell

## Step 6 — Expose publicly via ngrok (optional)

Skip if you only need to call the API from within the notebook.

In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok
# ngrok.set_auth_token("your_token")  # optional
tunnel = ngrok.connect(8000)
print(f"✅ Public URL: {tunnel.public_url}")

## Step 7 — Submit a checkpoint import job

`facebook/opt-125m` — small public model, no HuggingFace token required.

In [ ]:
import requests
import json

BASE = "http://localhost:8000"

resp = requests.post(f"{BASE}/v1/checkpoints/import", json={
    "source_path": "facebook/opt-125m",
    "target_name": "opt-125m-megatron",
    # "hf_token": "hf_..."  # uncomment for gated models
})

print("Response:", resp.status_code)
print(json.dumps(resp.json(), indent=2))

job_id = resp.json()["job_id"]
print(f"\nJob ID: {job_id}")

## Step 8 — Stream logs in real-time

In [ ]:
import websocket
import threading
import json

def stream_logs(job_id):
    ws = websocket.WebSocket()
    ws.connect(f"ws://localhost:8000/v1/ws/jobs/{job_id}/logs")
    print(f"--- Log stream for job {job_id} ---")
    while True:
        msg = ws.recv()
        if not msg:
            break
        try:
            data = json.loads(msg)
            if data.get("type") == "stream_end":
                print(f"\n--- Stream ended (status: {data['status']}) ---")
                break
        except json.JSONDecodeError:
            print(msg)
    ws.close()

log_thread = threading.Thread(target=stream_logs, args=(job_id,), daemon=True)
log_thread.start()
print("Log stream started in background thread.")

## Step 9 — Poll job status and GPU telemetry

In [ ]:
import time

while True:
    resp = requests.get(f"{BASE}/v1/jobs/{job_id}")
    job  = resp.json()
    status   = job["status"]
    progress = job.get("progress") or {}

    print(f"Status: {status}", end="")
    if progress:
        step  = progress.get("step", "?")
        total = progress.get("total_steps", "?")
        loss  = progress.get("loss", "?")
        gpus  = progress.get("gpus", [])
        print(f" | Step: {step}/{total} | Loss: {loss}", end="")
        if gpus:
            g = gpus[0]
            print(f" | GPU: {g.get('util_pct')}% | {g.get('mem_used_gb'):.1f}/{g.get('mem_total_gb'):.0f} GB | {g.get('temp_c')}°C", end="")
    print()

    if status in ("completed", "failed", "cancelled"):
        if status == "failed":
            print(f"\n❌ Error: {job.get('error')}")
        else:
            print(f"\n✅ Job {status} successfully")
        break
    time.sleep(5)

## Step 10 — View registered checkpoints

In [ ]:
resp = requests.get(f"{BASE}/v1/checkpoints")
print(json.dumps(resp.json(), indent=2))

## Step 11 — Launch a LoRA fine-tuning job (optional)

Requires a completed checkpoint from Step 7 and a dataset directory.

In [ ]:
resp = requests.post(f"{BASE}/v1/peft/lora", json={
    "pretrained_checkpoint": "/content/checkpoints/opt-125m-megatron",
    "dataset_path": "/content/datasets/my_data",
    "output_dir": "/content/checkpoints/opt-125m-lora",
    "num_gpus": 1,
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_target_modules": ["linear_qkv", "linear_proj"],
})
print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

## Cleanup

In [ ]:
server_proc.terminate()
server_proc.wait()
print("✅ API server stopped")
try:
    from pyngrok import ngrok
    ngrok.kill()
    print("✅ ngrok tunnel closed")
except Exception:
    pass

## Known Colab Compatibility Notes

| Issue | Cause | Fix applied in this notebook |
|---|---|---|
| `megatron-core[dev,mlm]` resolution conflict | Dev extras clash with Colab env | `megatron-bridge --no-deps`, then `megatron-core` separately |
| `libcublas.so.13: not found` | Driver reports CUDA 13.0 but toolkit is 12.8 | `core_cu12` variant |
| `undefined symbol: _ZNK3c104cuda10CUDAStream5queryEv` | PyTorch 2.10.0 too new for any pre-built TE wheel | Build transformer-engine from source (~20 min) |
| `transformer-engine==2.13.0` incompatible | megatron-bridge 0.3.1 requires `<2.13.0` | Build from `@stable` branch (currently 2.12.x) |
| `transformers==5.0.0` incompatible | megatron-bridge 0.3.1 requires `<5.0.0` | `pip install "transformers<5.0.0"` |
| Worker subprocess cannot find CUDA libs | `LD_LIBRARY_PATH` not inherited | Set before uvicorn start, `env=os.environ.copy()` in `Popen` |

**Production note:** All these issues are eliminated by using the
`nvcr.io/nvidia/nemo:25.02` NGC container as the worker base image.
That container has PyTorch, CUDA, and transformer-engine all pre-compiled
together — no ABI conflicts, no build-from-source. This is the recommended
path for actual training (see `docker/Dockerfile.worker` and `docs/deployment.md`).